# LeetCode 3161 — Block Placement Queries bằng Python

- Link: https://leetcode.com/problems/block-placement-queries/
- Độ khó: **Hard**
- Chủ đề: **Khoảng trống giữa các chướng ngại** lưu tại đầu phải của khoảng; truy vấn = `max(khoảng trống lớn nhất kết thúc ≤ x, x − chướng ngại gần nhất ≤ x)`; cấu trúc: **segment tree max + danh sách sắp xếp**, hoặc **xử lý ngược thời gian + DSU + Fenwick prefix-max**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def getResults(self, queries: List[List[int]]) -> List[bool]:
```

## 1. Mục tiêu học

1. Mô hình: các chướng ngại `0 = o₀ < o₁ < o₂ < …` (gốc 0 coi như một "chướng ngại"); khối cỡ `sz` đặt được trong `[0, x]` ⇔ có một đoạn trống dài `≥ sz` nằm trong `[0, x]`.
2. Các đoạn trống trong `[0, x]`: `[o_{i−1}, o_i]` với `o_i ≤ x`, và đoạn đuôi `[pred(x), x]`.
3. Lưu `gap[o_i] = o_i − o_{i−1}` tại **đầu phải** ⇒ "max các đoạn trống kết thúc `≤ x`" = **prefix max**.
4. Chèn chướng ngại `x` giữa `p` và `s`: đặt `gap[x] = x − p`, `gap[s] = s − x` ⇒ segment tree max cập nhật điểm.
5. Cách khác: đảo ngược thời gian — chèn thành **xoá**, khoảng trống chỉ **tăng** ⇒ Fenwick prefix-max + DSU tìm chướng ngại kề.

## 2. Đề bài & ví dụ

Trục số vô hạn bắt đầu từ 0. Hai loại truy vấn:

- `[1, x]`: xây chướng ngại tại `x` (đảm bảo chưa có).
- `[2, x, sz]`: có đặt được một khối dài `sz` nằm hoàn toàn trong `[0, x]` mà không **cắt qua** chướng ngại nào (được chạm) không? (Không đặt thật.)

Trả về danh sách `bool` cho các truy vấn loại 2.

**Ví dụ 1:** `[[1,2],[2,3,3],[2,3,1],[2,2,2]]` → `[false, true, true]`.

**Ví dụ 2:** `[[1,7],[2,7,6],[1,2],[2,7,5],[2,7,6]]` → `[true, true, false]`.

**Ràng buộc:** `1 ≤ queries.length ≤ 1.5·10⁵`; `1 ≤ x, sz ≤ min(5·10⁴, 3 · queries.length)`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right, insort
import random, time

examples = [([[1, 2], [2, 3, 3], [2, 3, 1], [2, 2, 2]], [False, True, True]),
            ([[1, 7], [2, 7, 6], [1, 2], [2, 7, 5], [2, 7, 6]], [True, True, False])]
for q, exp in examples:
    print(q, '-> kỳ vọng', exp)

[[1, 2], [2, 3, 3], [2, 3, 1], [2, 2, 2]] -> kỳ vọng [False, True, True]
[[1, 7], [2, 7, 6], [1, 2], [2, 7, 5], [2, 7, 6]] -> kỳ vọng [True, True, False]


## 3. Kiến thức nền

### 3.1. Đoạn trống dài nhất trong `[0, x]`

Với các chướng ngại đã sắp `0 = o₀ < o₁ < … < o_m`, đặt `p = pred(x)` = chướng ngại lớn nhất `≤ x`. Các đoạn trống tối đại nằm trong `[0, x]` là `[o_{i−1}, o_i]` (`o_i ≤ x`) và `[p, x]`. Khối dài `sz` đặt được ⇔ một trong các độ dài đó `≥ sz` (khối có thể chạm chướng ngại ở hai đầu).

In [2]:
def longest_free(obstacles_sorted, x):
    best = 0; prev = 0
    for o in obstacles_sorted:
        if o > x:
            break
        best = max(best, o - prev); prev = o
    return max(best, x - prev)

obs = [2, 7]
for x in (1, 2, 3, 7, 10):
    print(f'x = {x:2d}: đoạn trống dài nhất trong [0, x] = {longest_free(obs, x)}')

x =  1: đoạn trống dài nhất trong [0, x] = 1
x =  2: đoạn trống dài nhất trong [0, x] = 2
x =  3: đoạn trống dài nhất trong [0, x] = 2
x =  7: đoạn trống dài nhất trong [0, x] = 5
x = 10: đoạn trống dài nhất trong [0, x] = 5


### 3.2. Lưu khoảng trống ở đầu phải

`gap[o_i] = o_i − o_{i−1}`. Khi đó `max_{o_i ≤ x} gap[o_i]` là **prefix max** trên trục toạ độ tới `x` (các vị trí không phải chướng ngại có `gap = 0`). Chèn `x` giữa `p < x < s` chỉ đổi `gap[x]` (mới) và `gap[s]` (nhỏ đi).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung các chướng ngại là vạch trên thước. Mỗi khoảng giữa hai vạch là một "ngăn"; ghi độ dài ngăn vào vạch bên phải. Câu hỏi `[2, x, sz]`: "có ngăn nào (đầy đủ, nằm trước `x`) dài `≥ sz`, hoặc phần lẻ từ vạch cuối trước `x` tới `x` dài `≥ sz`?"

Ví dụ 2 sau khi có vạch `2, 7`: ngăn `[0, 2]` dài 2, ngăn `[2, 7]` dài 5 ⇒ `[2, 7, 5]` đúng, `[2, 7, 6]` sai.

## 5. Cách 1 — Brute Force: danh sách sắp xếp + quét

In [3]:
def brute(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            res.append(longest_free(obs, q[1]) >= q[2])
    return res

for q, exp in examples:
    print('brute:', brute(q), '| kỳ vọng', exp)

brute: [False, True, True] | kỳ vọng [False, True, True]
brute: [True, True, False] | kỳ vọng [True, True, False]


**Độ phức tạp:** `O(m)` mỗi truy vấn loại 2 ⇒ `O(q²)` xấu nhất (`2.25·10^{10}`).

## 6. Ý tưởng thuật toán / bước đột phá

```
obs = [0]                       # danh sách sắp xếp, có gốc 0
seg = segment tree max trên toạ độ 0..M
[1, x]: p = pred(x), s = succ(x)
        seg[x] = x − p; nếu có s: seg[s] = s − x
        insort(obs, x)
[2, x, sz]: p = pred(x); ok = max(seg.max(0..x), x − p) >= sz
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Segment tree max + `bisect.insort` (lời giải chính)

`insort` trên list Python là `O(m)` dịch bộ nhớ nhưng rất nhanh (memmove); với `m ≤ 5·10⁴` thực tế vài trăm ms. (Trên LeetCode có thể thay bằng `sortedcontainers.SortedList`.)

In [4]:
def segtree_online(queries):
    M = max(q[1] for q in queries) + 1
    size = 1
    while size < M + 1:
        size *= 2
    seg = [0] * (2 * size)
    def setv(i, v):
        i += size; seg[i] = v; i //= 2
        while i:
            seg[i] = max(seg[2 * i], seg[2 * i + 1]); i //= 2
    def prefix_max(r):                       # max seg[0..r]
        s = 0; l = size; r += size + 1
        while l < r:
            if l & 1: s = max(s, seg[l]); l += 1
            if r & 1: r -= 1; s = max(s, seg[r])
            l //= 2; r //= 2
        return s
    obs = [0]; res = []
    for q in queries:
        x = q[1]
        i = bisect_right(obs, x); p = obs[i - 1]
        if q[0] == 1:
            setv(x, x - p)
            if i < len(obs):
                s = obs[i]; setv(s, s - x)
            obs.insert(i, x)
        else:
            res.append(max(prefix_max(x), x - p) >= q[2])
    return res

for q, exp in examples:
    print('segment tree:', segtree_online(q), '| kỳ vọng', exp)

segment tree: [False, True, True] | kỳ vọng [False, True, True]
segment tree: [True, True, False] | kỳ vọng [True, True, False]


### 7.2. Cách 3 — Ngược thời gian: DSU + Fenwick prefix-max

Xây **tất cả** chướng ngại trước, duyệt truy vấn từ cuối lên: `[1, x]` trở thành **xoá** `x`. Khi xoá `x` (giữa `p` và `s`), khoảng `gap[s]` tăng thành `s − p` — giá trị chỉ **tăng** ⇒ Fenwick prefix-max dùng được. `gap[x]` cũ để nguyên (không xoá được trong Fenwick max), nhưng nó `= x − p' ≤ x − p` — luôn **không vượt** một ứng viên thật (đoạn đuôi hoặc `gap[s]`), nên không làm sai đáp án. `pred`/`succ` sau khi xoá tìm bằng DSU "nhảy qua phần tử đã xoá".

In [5]:
def offline_reverse(queries):
    M = max(q[1] for q in queries) + 2
    alive = [False] * (M + 1); alive[0] = True
    for q in queries:
        if q[0] == 1:
            alive[q[1]] = True
    # DSU tìm chướng ngại sống gần nhất về bên trái (<= i) và bên phải (>= i)
    left = list(range(M + 1)); right = list(range(M + 2))
    for i in range(1, M + 1):
        if not alive[i]:
            left[i] = i - 1
    for i in range(M + 1):
        if not alive[i] and i != 0:
            right[i] = i + 1
    def findL(i):
        r = i
        while left[r] != r:
            r = left[r]
        while left[i] != r:
            left[i], i = r, left[i]
        return r
    def findR(i):
        r = i
        while r <= M and right[r] != r:
            r = right[r]
        while i <= M and right[i] != r:
            right[i], i = r, right[i]
        return r                                   # > M nghĩa là không có
    bit = [0] * (M + 2)
    def upd(i, v):
        i += 1
        while i <= M + 1:
            if bit[i] < v: bit[i] = v
            i += i & -i
    def pref(i):
        i += 1; s = 0
        while i > 0:
            if bit[i] > s: s = bit[i]
            i -= i & -i
        return s
    prev = 0
    for i in range(1, M + 1):
        if alive[i]:
            upd(i, i - prev); prev = i
    out = []
    for q in reversed(queries):
        x = q[1]
        if q[0] == 1:
            alive[x] = False; left[x] = x - 1; right[x] = x + 1
            p = findL(x); s = findR(x)
            if s <= M:
                upd(s, s - p)
        else:
            p = findL(x)
            out.append(max(pref(x), x - p) >= q[2])
    return out[::-1]

for q, exp in examples:
    print('ngược thời gian:', offline_reverse(q), '| kỳ vọng', exp)

ngược thời gian: [False, True, True] | kỳ vọng [False, True, True]
ngược thời gian: [True, True, False] | kỳ vọng [True, True, False]


### 7.3. Cách 4 — Segment tree trên trục lưu (trái nhất, phải nhất, khoảng trống trong)

Mỗi nút đoạn `[L, R]` lưu: chướng ngại trái nhất `lo`, phải nhất `hi`, và khoảng trống lớn nhất **giữa hai chướng ngại** trong đoạn. Ghép: `inner = max(inner_L, inner_R, lo_R − hi_L)`. Truy vấn `[0, x]`: ghép các nút phủ `[0, x]` rồi lấy `max(inner, x − hi)` (gốc 0 luôn là chướng ngại). Không cần danh sách sắp xếp riêng.

In [6]:
def segtree_structural(queries):
    M = max(q[1] for q in queries) + 1
    size = 1
    while size < M + 1:
        size *= 2
    NONE = (None, None, 0)
    T = [NONE] * (2 * size)
    def comb(A, B):
        if A[0] is None: return B
        if B[0] is None: return A
        return (A[0], B[1], max(A[2], B[2], B[0] - A[1]))
    def setp(i):
        j = i + size; T[j] = (i, i, 0); j //= 2
        while j:
            T[j] = comb(T[2 * j], T[2 * j + 1]); j //= 2
    setp(0)
    res = []
    for q in queries:
        if q[0] == 1:
            setp(q[1])
        else:
            x = q[1]; l = size; r = x + size + 1; left = NONE; rights = []
            while l < r:
                if l & 1: left = comb(left, T[l]); l += 1
                if r & 1: r -= 1; rights.append(T[r])
                l //= 2; r //= 2
            for node in reversed(rights):
                left = comb(left, node)
            res.append(max(left[2], x - left[1]) >= q[2])
    return res

for q, exp in examples:
    print('segment tree cấu trúc:', segtree_structural(q), '| kỳ vọng', exp)

segment tree cấu trúc: [False, True, True] | kỳ vọng [False, True, True]
segment tree cấu trúc: [True, True, False] | kỳ vọng [True, True, False]


## 8. TRACE từng bước

Trace cách 2: in danh sách chướng ngại, các `gap` khác 0 và lời giải từng truy vấn.

In [7]:
def trace(queries):
    obs = [0]; gap = {}
    for q in queries:
        x = q[1]; i = bisect_right(obs, x); p = obs[i - 1]
        if q[0] == 1:
            gap[x] = x - p
            if i < len(obs):
                gap[obs[i]] = obs[i] - x
            obs.insert(i, x)
            print(f'[1, {x}]: chướng ngại {obs} | gap = { {k: gap[k] for k in sorted(gap)} }')
        else:
            inner = max([g for k, g in gap.items() if k <= x], default=0)
            print(f'[2, {x}, {q[2]}]: max gap ≤ x = {inner}, đuôi x − pred = {x} − {p} = {x - p} -> {max(inner, x - p) >= q[2]}')

trace([[1, 5], [1, 12], [2, 11, 7], [2, 12, 7], [1, 8], [2, 12, 7], [2, 20, 8]])

[1, 5]: chướng ngại [0, 5] | gap = {5: 5}
[1, 12]: chướng ngại [0, 5, 12] | gap = {5: 5, 12: 7}
[2, 11, 7]: max gap ≤ x = 5, đuôi x − pred = 11 − 5 = 6 -> False
[2, 12, 7]: max gap ≤ x = 7, đuôi x − pred = 12 − 12 = 0 -> True
[1, 8]: chướng ngại [0, 5, 8, 12] | gap = {5: 5, 8: 3, 12: 4}
[2, 12, 7]: max gap ≤ x = 5, đuôi x − pred = 12 − 12 = 0 -> False
[2, 20, 8]: max gap ≤ x = 5, đuôi x − pred = 20 − 12 = 8 -> True


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def getResults(self, queries: List[List[int]]) -> List[bool]:
        M = 0
        for q in queries:
            if q[1] > M:
                M = q[1]
        size = 1
        while size < M + 1:
            size *= 2
        seg = [0] * (2 * size)
        obs = [0]
        res = []
        for q in queries:
            x = q[1]
            i = bisect_right(obs, x)
            p = obs[i - 1]
            if q[0] == 1:
                j = x + size
                seg[j] = x - p
                j >>= 1
                while j:
                    a, b = seg[2 * j], seg[2 * j + 1]
                    seg[j] = a if a > b else b
                    j >>= 1
                if i < len(obs):
                    s = obs[i]
                    j = s + size
                    seg[j] = s - x
                    j >>= 1
                    while j:
                        a, b = seg[2 * j], seg[2 * j + 1]
                        seg[j] = a if a > b else b
                        j >>= 1
                obs.insert(i, x)
            else:
                best = x - p
                l = size
                r = x + size + 1
                while l < r:
                    if l & 1:
                        if seg[l] > best:
                            best = seg[l]
                        l += 1
                    if r & 1:
                        r -= 1
                        if seg[r] > best:
                            best = seg[r]
                    l >>= 1
                    r >>= 1
                res.append(best >= q[2])
        return res

for q, exp in examples:
    got = Solution().getResults(q)
    print(got, 'OK' if got == exp else 'SAI')

[False, True, True] OK
[True, True, False] OK


## 10. Vì sao thuật toán đúng

1. **Bất biến:** với mọi chướng ngại `o` (khác 0), `seg[o] = o − pred(o)` (khoảng trống ngay bên trái); vị trí khác có `seg = 0`. Chèn `x` giữa `p` và `s`: chỉ khoảng bên trái `x` (mới) và bên trái `s` (bị chia đôi) thay đổi — ta cập nhật đúng hai giá trị này.
2. **Truy vấn:** các đoạn trống tối đại trong `[0, x]` là các khoảng kết thúc tại chướng ngại `o ≤ x` (độ dài `seg[o]`) cùng đoạn đuôi `[pred(x), x]`. Một khoảng kết thúc tại `o > x` không nằm trọn trong `[0, x]`; phần của nó trong `[0, x]` chính là đoạn đuôi. Khối đặt được ⇔ một đoạn có độ dài `≥ sz` ⇔ `max(seg[0..x], x − pred(x)) ≥ sz`.
3. **Cách 3:** giá trị Fenwick chỉ tăng; giá trị cũ `seg[x]` của chướng ngại đã xoá `≤` đoạn trống thật chứa nó ⇒ max không đổi.

## 11. Phân tích Time / Space Complexity

| Cách | Loại 1 | Loại 2 | Space |
|---|---|---|---|
| 1. Brute | `O(m)` | `O(m)` | `O(m)` |
| 2. Segment tree + insort | `O(log M + m)` (memmove) | `O(log M)` | `O(M)` |
| 3. Ngược thời gian | `O(α + log M)` | `O(log M)` | `O(M)` |
| 4. Segment tree cấu trúc | `O(log M)` | `O(log M)` | `O(M)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'segtree': segtree_online, 'reverse': offline_reverse, 'structural': segtree_structural,
         'submit': lambda q: Solution().getResults(q)}
edge = [([[2, 3, 3]], [True]), ([[2, 3, 4]], [False]), ([[1, 1], [2, 1, 1], [2, 2, 2]], [True, False]),
        ([[1, 10], [2, 5, 6]], [False]), ([[1, 2], [2, 5, 3]], [True]), ([[1, 4], [1, 6], [2, 5, 4]], [True]),
        ([[1, 3], [1, 1], [1, 2], [2, 3, 2], [2, 4, 1]], [False, True])]
for q, exp in examples + edge:
    assert brute(q) == exp, (q, brute(q))
    for name, f in impls.items():
        assert f(q) == exp, (name, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[[1, 10], [2, 5, 6]]` — khoảng `[0, 10]` dài 10 nhưng chỉ phần `[0, 5]` nằm trong phạm vi ⇒ `False`. `[[1, 4], [1, 6], [2, 5, 4]]` — ngăn `[0, 4]` dài đúng 4 (tính từ gốc 0) ⇒ `True`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3161)
def rand_queries(q, M):
    used = set(); out = []
    for _ in range(q):
        if rng.random() < 0.5 and len(used) < M:
            x = rng.randint(1, M)
            while x in used:
                x = rng.randint(1, M)
            used.add(x); out.append([1, x])
        else:
            out.append([2, rng.randint(1, M), rng.randint(1, M)])
    if all(t[0] == 1 for t in out):
        out.append([2, rng.randint(1, M), rng.randint(1, M)])
    return out

cnt = 0
for _ in range(400):
    qs = rand_queries(rng.randint(1, 15), rng.choice([5, 10, 30]))
    exp = brute(qs)
    for name, f in impls.items():
        assert f(qs) == exp, (name, qs)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 400 bộ ngẫu nhiên.


### 12.3. Ca lớn `1.5·10⁵` truy vấn, toạ độ tới `5·10⁴`

In [11]:
Q = 15 * 10 ** 4; M = 5 * 10 ** 4
xs = rng.sample(range(1, M + 1), M)
big = []
it = iter(xs)
for t in range(Q):
    if t % 3 == 0:
        big.append([1, next(it)])
    else:
        big.append([2, rng.randint(1, M), rng.randint(1, 50)])
t0 = time.time(); x = Solution().getResults(big); t1 = time.time() - t0
t0 = time.time(); y = offline_reverse(big); t2 = time.time() - t0
assert x == y
print(f'{Q} truy vấn: {sum(x)} True / {len(x)} | nộp {t1:.2f}s | ngược thời gian {t2:.2f}s')

150000 truy vấn: 41572 True / 100000 | nộp 0.40s | ngược thời gian 0.21s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đoạn đuôi `[pred(x), x]`

In [12]:
def wrong_no_tail(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            best = 0; prev = 0
            for o in obs:
                if o > q[1]: break
                best = max(best, o - prev); prev = o
            res.append(best >= q[2])                                   # SAI: thiếu x - prev
    return res

qs = [[1, 2], [2, 5, 3]]
print('sai :', wrong_no_tail(qs), '| đúng:', Solution().getResults(qs))

sai : [False] | đúng: [True]


**Vì sao sai:** sau chướng ngại cuối cùng `≤ x` vẫn còn khoảng trống tới `x` (ở đây `[2, 5]` dài 3).

### Lỗi 2 — Lấy max **mọi** khoảng trống (kể cả khoảng vượt quá `x`)

In [13]:
def wrong_all_gaps(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            pts = [0] + obs
            best = max([pts[i] - pts[i - 1] for i in range(1, len(pts))], default=0)   # SAI: không giới hạn bởi x
            prev = max([o for o in pts if o <= q[1]])
            res.append(max(best, q[1] - prev) >= q[2])
    return res

qs = [[1, 10], [2, 5, 6]]
print('sai :', wrong_all_gaps(qs), '| đúng:', Solution().getResults(qs))

sai : [True] | đúng: [False]


**Vì sao sai:** khoảng `[0, 10]` dài 10 nhưng khối phải nằm trong `[0, 5]`.

### Lỗi 3 — So sánh ngặt `> sz`

In [14]:
def wrong_strict(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            res.append(longest_free(obs, q[1]) > q[2])                  # SAI: phải là >=
    return res

qs = [[2, 3, 3]]
print('sai :', wrong_strict(qs), '| đúng:', Solution().getResults(qs))

sai : [False] | đúng: [True]


**Vì sao sai:** khối **được chạm** chướng ngại (và chạm hai đầu `0`, `x`) ⇒ khoảng dài đúng `sz` vẫn đặt được.

### Lỗi 4 — Không coi gốc 0 là "chướng ngại" (bỏ khoảng đầu tiên)

In [15]:
def wrong_no_origin(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            pts = [o for o in obs if o <= q[1]]
            best = max([pts[i] - pts[i - 1] for i in range(1, len(pts))], default=0)   # SAI: thiếu [0, o1]
            tail = q[1] - pts[-1] if pts else q[1]
            res.append(max(best, tail) >= q[2])
    return res

qs = [[1, 4], [1, 6], [2, 5, 4]]
print('sai :', wrong_no_origin(qs), '| đúng:', Solution().getResults(qs))

sai : [False] | đúng: [True]


**Vì sao sai:** đoạn `[0, o₁]` là một ngăn hợp lệ — dễ quên khi chỉ lấy hiệu giữa các chướng ngại.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn trống dài nhất" + chèn điểm ⇒ lưu độ dài khoảng ở **một đầu** (đầu phải) để biến thành bài max tiền tố.
- Chèn-only mà cấu trúc không xoá được ⇒ thử **đảo thời gian** (chèn ↔ xoá) để giá trị đơn điệu.
- Gốc 0 / biên là "chướng ngại ảo".

## 15. Biến thể của bài

1. Hỏi **độ dài khối lớn nhất** đặt được trong `[0, x]` ⇒ trả `max(...)` thay vì so sánh.
2. Có cả **xoá** chướng ngại online ⇒ segment tree cấu trúc (cách 4) vẫn hoạt động (đặt lá về `None`).
3. Phạm vi `[a, b]` thay vì `[0, b]` ⇒ cách 4 truy vấn đoạn, thêm hai đoạn biên.
4. Nhiều khối cùng lúc (đặt `k` khối không chồng nhau) ⇒ cần đếm số khối chứa được: `Σ ⌊gap / sz⌋` — khó hơn nhiều.

In [16]:
# Biến thể 2: có thêm xoá [3, x] — dùng segment tree cấu trúc
def with_delete(queries):
    M = max(q[1] for q in queries) + 1; size = 1
    while size < M + 1:
        size *= 2
    NONE = (None, None, 0); T = [NONE] * (2 * size)
    def comb(A, B):
        if A[0] is None: return B
        if B[0] is None: return A
        return (A[0], B[1], max(A[2], B[2], B[0] - A[1]))
    def setp(i, on):
        j = i + size; T[j] = (i, i, 0) if on else NONE; j //= 2
        while j:
            T[j] = comb(T[2 * j], T[2 * j + 1]); j //= 2
    setp(0, True); res = []
    for q in queries:
        if q[0] == 1: setp(q[1], True)
        elif q[0] == 3: setp(q[1], False)
        else:
            x = q[1]; l = size; r = x + size + 1; left = NONE; rights = []
            while l < r:
                if l & 1: left = comb(left, T[l]); l += 1
                if r & 1: r -= 1; rights.append(T[r])
                l //= 2; r //= 2
            for node in reversed(rights):
                left = comb(left, node)
            res.append(max(left[2], x - left[1]) >= q[2])
    return res

def with_delete_brute(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1: insort(obs, q[1])
        elif q[0] == 3: obs.remove(q[1])
        else: res.append(longest_free(obs, q[1]) >= q[2])
    return res

for _ in range(200):
    present = set(); qs = []
    for _ in range(rng.randint(1, 15)):
        r = rng.random()
        if r < 0.35:
            x = rng.randint(1, 12)
            if x not in present: present.add(x); qs.append([1, x])
        elif r < 0.55 and present:
            x = rng.choice(sorted(present)); present.remove(x); qs.append([3, x])
        else:
            qs.append([2, rng.randint(1, 12), rng.randint(1, 12)])
    if any(q[0] == 2 for q in qs):
        assert with_delete(qs) == with_delete_brute(qs)
print('Biến thể có xoá: segment tree cấu trúc khớp brute.')

Biến thể có xoá: segment tree cấu trúc khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Ghi chú |
|---|---|---|---|
| 1 | Danh sách + quét | `O(m)` | Oracle |
| 2 | Seg tree max + insort | `O(log M)` (+ memmove) | **Nộp bài** |
| 3 | Ngược thời gian, DSU + Fenwick max | `O(log M)` | Không cần cấu trúc sắp xếp |
| 4 | Seg tree (lo, hi, inner) | `O(log M)` | Hỗ trợ cả xoá |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1, 3], [2, 3, 3], [2, 4, 2]]`.

**Bài 2 (tính tay):** `[[1, 5], [1, 1], [2, 5, 4], [2, 9, 4], [2, 9, 5]]`.

**Bài 3 (điền TODO):** biến thể 1 — trả về độ dài khối **lớn nhất** đặt được cho mỗi truy vấn loại 2 (`[2, x]`).

**Bài 4 (tự viết lại):** phạm vi `[a, b]`: truy vấn `[2, a, b, sz]`.

**Bài 5 (mini challenge):** đếm số vị trí bắt đầu **nguyên** `t` sao cho khối `[t, t + sz]` đặt được trong `[0, x]`.

In [17]:
def max_block_todo(queries):
    # TODO
    raise NotImplementedError

def range_ab_todo(queries):
    # TODO
    raise NotImplementedError

def count_positions_todo(queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ngăn `[0, 3]` dài 3. (2) `[2, 3, 3]`: đúng. `[2, 4, 2]`: ngăn `[0, 3]` dài 3 ≥ 2. (3) `[True, True]`.

**Bài 2.** (1) Chướng ngại `1, 5`: ngăn `[0,1]`, `[1,5]` dài 4. (2) `[2,5,4]` đúng; `[2,9,4]` đúng (đuôi `[5, 9]` cũng dài 4); `[2,9,5]` sai. (3) `[True, True, False]`.

**Bài 3.** (1) Bỏ phép so sánh. (2) Trả `max(seg.prefix_max(x), x − pred(x))`. (3) Kiểm bằng `longest_free`.

**Bài 4.** (1) Dùng cách 4 trên đoạn `[a, b]`. (2) Thêm hai đoạn biên `[a, lo]` và `[hi, b]` (hoặc cả `[a, b]` nếu không có chướng ngại bên trong). (3) Kiểm bằng brute.

**Bài 5.** (1) Mỗi đoạn trống dài `L ≥ sz` cho `L − sz + 1` vị trí nguyên. (2) Cộng các đoạn trong `[0, x]` (gồm đoạn đuôi). (3) Brute: thử mọi `t ∈ [0, x − sz]`.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().getResults([[1, 3], [2, 3, 3], [2, 4, 2]]) == [True, True]
assert Solution().getResults([[1, 5], [1, 1], [2, 5, 4], [2, 9, 4], [2, 9, 5]]) == [True, True, False]

def max_block(queries):
    M = max(q[1] for q in queries) + 1; size = 1
    while size < M + 1:
        size *= 2
    seg = [0] * (2 * size); obs = [0]; res = []
    def setv(i, v):
        i += size; seg[i] = v; i //= 2
        while i:
            seg[i] = max(seg[2 * i], seg[2 * i + 1]); i //= 2
    def pm(r):
        s = 0; l = size; r += size + 1
        while l < r:
            if l & 1: s = max(s, seg[l]); l += 1
            if r & 1: r -= 1; s = max(s, seg[r])
            l //= 2; r //= 2
        return s
    for q in queries:
        x = q[1]; i = bisect_right(obs, x); p = obs[i - 1]
        if q[0] == 1:
            setv(x, x - p)
            if i < len(obs): setv(obs[i], obs[i] - x)
            obs.insert(i, x)
        else:
            res.append(max(pm(x), x - p))
    return res

for _ in range(200):
    qs = rand_queries(rng.randint(1, 15), 20); obs = []; exp = []
    for q in qs:
        if q[0] == 1: insort(obs, q[1])
        else: exp.append(longest_free(obs, q[1]))
    assert max_block(qs) == exp
print('Bài 1, 2 đúng | Bài 3: khối lớn nhất khớp brute.')

Bài 1, 2 đúng | Bài 3: khối lớn nhất khớp brute.


In [19]:
# Bài 4, 5 (bản đơn giản, kiểm bằng brute)
def range_ab(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            _, a, b, sz = q
            inside = obs[bisect_right(obs, a): bisect_left(obs, b)]
            pts = [a] + inside + [b]
            res.append(max(pts[i] - pts[i - 1] for i in range(1, len(pts))) >= sz)
    return res

def range_ab_brute(queries):
    obs = set(); res = []
    for q in queries:
        if q[0] == 1:
            obs.add(q[1])
        else:
            _, a, b, sz = q
            ok = any(all(not (t < o < t + sz) for o in obs) for t in range(a, b - sz + 1))
            res.append(ok)
    return res

def count_positions(queries):
    obs = []; res = []
    for q in queries:
        if q[0] == 1:
            insort(obs, q[1])
        else:
            x, sz = q[1], q[2]; prev = 0; tot = 0
            for o in obs + [None]:
                end = x if o is None or o > x else o
                L = end - prev
                if L >= sz: tot += L - sz + 1
                if o is None or o > x: break
                prev = o
            res.append(tot)
    return res

def count_positions_brute(queries):
    obs = set(); res = []
    for q in queries:
        if q[0] == 1: obs.add(q[1])
        else:
            x, sz = q[1], q[2]
            res.append(sum(1 for t in range(0, x - sz + 1) if all(not (t < o < t + sz) for o in obs)))
    return res

for _ in range(200):
    qs = rand_queries(rng.randint(1, 12), 15)
    assert count_positions(qs) == count_positions_brute(qs)
    qs4 = []
    for q in qs:
        if q[0] == 1: qs4.append(q)
        else:
            a = rng.randint(0, q[1]); qs4.append([2, a, q[1], rng.randint(1, max(1, q[1] - a + 1))])
    assert range_ab(qs4) == range_ab_brute(qs4)
print('Bài 4 (phạm vi [a, b]) và Bài 5 (đếm vị trí nguyên) khớp brute.')

Bài 4 (phạm vi [a, b]) và Bài 5 (đếm vị trí nguyên) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
obs = [0]; seg = max-segment-tree trên 0..M
[1, x]: i = bisect_right(obs, x); p = obs[i-1]
        seg[x] = x - p; nếu i < len(obs): seg[obs[i]] = obs[i] - x; obs.insert(i, x)
[2, x, sz]: p = obs[bisect_right(obs, x) - 1]
        ok = max(seg.max(0..x), x - p) >= sz
```

**Checklist:**

- [ ] Có gốc `0` trong danh sách chướng ngại?
- [ ] Cộng đoạn đuôi `x − pred(x)`?
- [ ] Chỉ max các khoảng kết thúc `≤ x`?
- [ ] So sánh `≥ sz` (được chạm)?
- [ ] Cập nhật cả khoảng của chướng ngại **kế tiếp** khi chèn?